<b>In Domain Query Template</b>
```python
STROKE_TEMPLATE = {
    "player_stroke_area": [
        "When does the {player} hits a {stroke} {hit_area}?",
        "At what point in the rally does the {player} perform a {stroke} {hit_area}?",
        "Identify when the {player} executes a {stroke} {hit_area}.",
        "Locate the moment when a {stroke} {hit_area} is played by the {player}."
    ],
    "player_stroke": [
        "When does the {player} hits a {stroke}?",
        "At what time does the {player} execute a {stroke}?",
        "When in this rally does the {player} perform a {stroke}?",
        "Identify the moment the {player} plays a {stroke}."
    ],
    "stroke_only": [
        "When is a {stroke} hits?",
        "At what moment does a {stroke} occur?",
        "When during the rally does a {stroke} happen?",
        "Locate when the {stroke} takes place."
    ],
    "hit_area_only": [
        "Which stroke is hit {hit_area}?",
        "Which shot lands {hit_area}?",
        "Identify the stroke that occurs {hit_area}.",
        "What stroke happens {hit_area}?"
    ]
}
STRATEGIES_TEMPLATE = {
    "four_corner": [
        "When does the players execute the four-corner pattern during a rally?",
        "Identify the timestamps where a corner-to-corner tactic occurs.",
        "At which points in the match do both backcourt corners get targeted in succession?"
    ],
    "net_shot": [
        "When do we see consecutive net-play exchanges in the rally?",
        "Locate moments of close-to-the-net shot sequences.",
        "At what times do players perform back-and-forth shots right at the net?"
    ],
    "back_court": [
        "When does a deep-court attack pattern appear?",
        "Pinpoint the instances of back-court rallies.",
        "At which moments are both rear-court zones hit consecutively?"
    ],
    "flat_shot_sequence": [
        "When do we observe a series of flat-drive strokes?",
        "Identify the moments of back-to-back flat-shot exchanges.",
        "At what points does a continuous flat-shot sequence unfold?"
    ],
    "upper_player_counter_attack": [
        "When does the upper player launch a counter-attack after a defensive shot?",
        "Identify moments where the upper player switches from defense to offense.",
        "At which timestamps does the upper player execute a defensive counterstrike?"
    ],
    "bottom_player_counter_attack": [
        "When does the bottom player initiate a counter-attack following a defensive return?",
        "Locate the points where the bottom player transitions from defense into an aggressive shot.",
        "At what moments does the bottom player perform a defensive riposte into attack?"
    ]
}
```

In [ ]:
import re
import os
import torch
import argparse
from lavis.common.config import Config
import lavis.tasks as tasks
from lavis.common.dist_utils import init_distributed_mode
from lavis.common.registry import registry
import warnings 

warnings.filterwarnings("ignore", category=FutureWarning)
def extract_ans(text: str) -> tuple[list[int], str]:
    thinking_match = re.search(r"<thinking>(.*?)</thinking>", text, flags=re.DOTALL)
    thinking_str = thinking_match.group(1).strip() if thinking_match else ""

    m = re.search(r"<answer>(.*?)</answer>", text, flags=re.DOTALL)
    if m:
        nums = re.findall(r"\b\d+\b", m.group(1))
        return list(map(int, nums)), thinking_str

    m2 = re.search(r"The event happens at strokes? ([\d,]+)", text)
    if m2:
        nums = m2.group(1).split(",")
        return [int(n) for n in nums], thinking_str

    return [], thinking_str


## Single Inference

In [ ]:


INSTRUCTION = (
    "<Video> This video has {n} strokes. "
    "You must answer based only on the strokes you see—do not invent or hallucinate any events. "
    "Let's think step by step. "
    "If the event occurs, output exactly “The event happens at strokes i,j,…” to list the stroke indices"
    "If the event does not occur, output exactly “The event does not occur”"
    )
@torch.no_grad()
def single_inference(
    clip_paths,   # List[str] clip 路徑
    question: str,
    cfg_path: str = "lavis/projects/instructblip/inference/inference_instructblip_badminton_qa_coT_3.yaml",
    device: str = "cuda"
):
    # 1. 載入 config 與 task
    args = argparse.Namespace(cfg_path=cfg_path, options=None, cfg_options=None)
    cfg = Config(args)
    init_distributed_mode(cfg.run_cfg)
    task = tasks.setup_task(cfg)

    proc_cfg = cfg.datasets_cfg["badminton_qa"].vis_processor.eval
    vis_processor = registry.get_processor_class(proc_cfg.name).from_config(proc_cfg)
    txt_proc_cfg = cfg.datasets_cfg["badminton_qa"].text_processor.eval
    text_processor = registry.get_processor_class(txt_proc_cfg.name).from_config(txt_proc_cfg)

    clip_tensors = []
    for p in clip_paths:
        clip_tensor = vis_processor(p)
        if clip_tensor.dim() != 4:
            raise ValueError(f"Processor output must be [T,C,H,W], got {clip_tensor.shape}")
        clip_tensors.append(clip_tensor)

    K = len(clip_tensors)
    T, C, H, W = clip_tensors[0].shape
    for t in clip_tensors:
        if t.shape != (T, C, H, W):
            raise ValueError(f"Inconsistent clip shapes: {t.shape} vs {(T,C,H,W)}")

    images_tensor = torch.stack(clip_tensors, dim=0).unsqueeze(0).to(device)  # [1, K, T, C, H, W]
    clip_mask = torch.ones(1, K, dtype=torch.bool, device=device)             # [1, K]

    text_input = text_processor(f"{INSTRUCTION} Question: {question} Answer:")
    qformer_instruction = text_processor("<Video> A short video description:")

    samples = {
        "images": images_tensor,
        "clip_mask": clip_mask,
        "text_input": [text_input],
        "Qformer_instruction": [qformer_instruction],
    }
    model = task.build_model(cfg).to(device)
    model.eval()
    outputs = model.predict_answers(
        samples,
        num_beams=5,
        inference_method="generate",
        max_len=300,
        min_len=30,
        length_penalty=0.0
    )

    return outputs, samples

In [ ]:

video_root = "lavis/configs/datasets/badminton_caption/input/images"
clips = [
    "game1_set1_26511.mp4",
    "game1_set1_26531.mp4",
    "game1_set1_26558.mp4",
    "game1_set1_26570.mp4",
    "game1_set1_26591.mp4",
    "game1_set1_26612.mp4",
    "game1_set1_26645.mp4",
    "game1_set1_26662.mp4",
    "game1_set1_26683.mp4"
]
clip_paths = [os.path.join(video_root, clip) for clip in clips]
question = "When do we see consecutive net-play exchanges in the rally?"
outputs, samples = single_inference(clip_paths, question)
from IPython.display import Video, display

for output in outputs:
    ans, thinking = extract_ans(output)
    video_path = [clip_paths[i] for i in ans]
    print(f"Thinking: {thinking}")
    print(f"Video Paths: {video_path}")

    for vp in video_path:
        display(Video(vp, embed=True))

## Batch Inference


In [ ]:
INSTRUCTION = (
    "<Video> This video has {n} strokes. "
    "You must answer based only on the strokes you see—do not invent or hallucinate any events. "
    "Let's think step by step. "
    "If the event occurs, output exactly “The event happens at strokes i,j,…” to list the stroke indices"
    "If the event does not occur, output exactly “The event does not occur”"
    )
@torch.no_grad()
def batch_inference(
    items,         
    question,
    cfg_path: str = "lavis/projects/instructblip/inference/inference_instructblip_badminton_qa_coT_3.yaml",
    device: str = "cuda"
):
    args = argparse.Namespace(cfg_path=cfg_path, options=None, cfg_options=None)
    cfg = Config(args)
    init_distributed_mode(cfg.run_cfg)
    task = tasks.setup_task(cfg)

    proc_cfg = cfg.datasets_cfg["badminton_qa"].vis_processor.eval
    vis_processor = registry.get_processor_class(proc_cfg.name).from_config(proc_cfg)
    txt_proc_cfg = cfg.datasets_cfg["badminton_qa"].text_processor.eval
    text_processor = registry.get_processor_class(txt_proc_cfg.name).from_config(txt_proc_cfg)
    processed_question = text_processor(f"{INSTRUCTION} Question: {question} Answer:")
    processed_qformer_instruction = text_processor("<Video> A short video description:")
    model = task.build_model(cfg).to(device)
    model.eval()

    per_sample_clips = []
    text_inputs, qformer_instructions = [], []
    qids, chunks, impossibles, answers = [], [], [], []

    for i, clip_paths in enumerate(items):
        clip_tensors = []
        for p in clip_paths:
            clip_tensor = vis_processor(p)  
            if not isinstance(clip_tensor, torch.Tensor) or clip_tensor.dim() != 4:
                raise ValueError(f"Processor output 必須是 [T,C,H,W]，但 {p} 得到 {None if not isinstance(clip_tensor, torch.Tensor) else clip_tensor.shape}")
            clip_tensors.append(clip_tensor)

        T, C, H, W = clip_tensors[0].shape
        for t in clip_tensors:
            if t.shape != (T, C, H, W):
                raise ValueError(f"同一樣本內 clip 尺寸不一致：{t.shape} vs {(T,C,H,W)}。請在 processor 端對齊或在此處理 padding。")

        per_sample_clips.append(torch.stack(clip_tensors, dim=0)) 


        text_inputs.append(processed_question)
        qformer_instructions.append(processed_qformer_instruction)

    ks = [x.size(0) for x in per_sample_clips]
    K_max = max(ks)

    padded, clip_masks = [], []
    for img, k_i in zip(per_sample_clips, ks):
        mask = torch.tensor([1]*k_i + [0]*(K_max - k_i), dtype=torch.bool)
        clip_masks.append(mask)

        if k_i < K_max:
            T, C, H, W = img.shape[1:]
            pad = torch.zeros((K_max - k_i, T, C, H, W), dtype=img.dtype, device=img.device)
            img = torch.cat([img, pad], dim=0)
        padded.append(img)

    # 4) 組 batch tensors
    batch_images = torch.stack(padded, dim=0).to(device)  
    clip_mask = torch.stack(clip_masks, dim=0).to(device) 

    samples = {
        "images": batch_images,
        "clip_mask": clip_mask,
        "text_input": text_inputs,              
        "Qformer_instruction": qformer_instructions,  
    }

    outputs = model.predict_answers(
        samples,
        num_beams=5,
        inference_method="generate",
        max_len=300,
        min_len=30,
        length_penalty=0.0
    )

    return outputs, samples

In [ ]:
items = [
     [
      "game48_set1_39414.mp4",
      "game48_set1_39431.mp4",
      "game48_set1_39449.mp4",
      "game48_set1_39473.mp4",
      "game48_set1_39499.mp4",
      "game48_set1_39518.mp4"
    ],
    [
      "game48_set2_55573.mp4",
      "game48_set2_55592.mp4",
      "game48_set2_55619.mp4",
      "game48_set2_55642.mp4",
      "game48_set2_55654.mp4"
    ],
]
question= "What stroke happens in the middle?"
video_root = "lavis/configs/datasets/badminton_caption/input/images"
items = [[os.path.join(video_root, it) for it in items[i]] for i in range(len(items))]
outputs, samples = batch_inference(items ,question)
from IPython.display import Video, display

for i, output in enumerate(outputs):
    ans, thinking = extract_ans(output)
    video_paths = [items[i][j] for j in ans]  
    
    print(f"Thinking: {thinking}")
    print(f"Video Paths: {video_paths}")
    
    for vp in video_paths:
        display(Video(vp, embed=True))  # 在 notebook 內嵌播放
